# Social Media Engagement Analytics Using Python
**Data Analytics (DA) – Module-End 5 Assignment: Python for Data Analysis**

Dataset: `social_media_engagement_5000.csv`

**Contents:** Task 1 Import & Setup · Task 2 Cleaning · Task 3 Exploration · Task 4 Wrangling · Task 5 Statistics · Task 6 Visualization (Matplotlib, Seaborn, Plotly) · Final Insights

## Task 1 — Data Import & Setup

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import plotly.express as px
    PLOTLY_OK = True
except ImportError:
    PLOTLY_OK = False
    print("plotly not installed -> run: pip install plotly")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
# Import the CSV using Pandas (GitHub link first, local file as a fallback)
URL = "https://raw.githubusercontent.com/GeethaGunasekaran1/Dataset_rep/main/social_media_engagement_5000.csv"
LOCAL = "social_media_engagement_5000.csv"

try:
    df = pd.read_csv(URL)
    print("Loaded from GitHub")
except Exception:
    df = pd.read_csv(LOCAL)
    print("Loaded from local file")

# Standardise column names: lowercase, underscores
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^0-9a-z]+", "_", regex=True).str.strip("_"))

# Map common alternative names to the names used in this notebook.
# If your CSV uses other names, add them to the lists below.
ALIASES = {
    "likes": ["likes", "like_count", "num_likes"],
    "comments": ["comments", "comment_count", "num_comments"],
    "shares": ["shares", "share_count", "num_shares"],
    "impressions": ["impressions", "reach", "views"],
    "watch_time": ["watch_time", "watch_time_seconds", "watch_time_sec", "avg_watch_time"],
    "engagement_rate": ["engagement_rate", "eng_rate"],
    "followers": ["followers", "follower_count", "followers_count"],
    "post_type": ["post_type", "content_type", "type"],
    "category": ["category", "content_category", "topic"],
    "country": ["country", "location", "region"],
    "gender": ["gender", "user_gender"],
    "age": ["age", "user_age"],
    "verified": ["verified", "is_verified", "verified_account"],
    "device": ["device", "device_type"],
    "sentiment": ["sentiment", "sentiment_label"],
    "hashtags": ["hashtags", "hashtag", "tags"],
    "hashtag_count": ["hashtag_count", "hashtags_count", "num_hashtags"],
    "date": ["date", "post_date", "timestamp", "post_time", "created_at", "datetime"],
    "hour": ["hour", "post_hour", "hour_of_day"],
    "time_of_day": ["time_of_day", "daypart"],
}
for canon, alts in ALIASES.items():
    if canon in df.columns:
        continue
    for a in alts:
        if a in df.columns:
            df = df.rename(columns={a: canon})
            break

def has(*cols):
    """True if all columns exist (lets the notebook skip a step instead of crashing)."""
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"Skipped - column(s) not in dataset: {missing}")
        return False
    return True

print("Columns:", list(df.columns))
df.head()

In [ ]:
# Check data types
print(df.dtypes)

# Convert date columns to datetime
for c in df.columns:
    if any(k in c for k in ("date", "timestamp", "datetime", "created_at", "post_time")):
        df[c] = pd.to_datetime(df[c], errors="coerce")
        print("Converted to datetime:", c)

# Convert numeric columns that were read as text
NUMERIC = ["likes", "comments", "shares", "impressions", "watch_time",
           "engagement_rate", "followers", "age", "hashtag_count"]
for c in NUMERIC:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

print()
print(df.dtypes)

In [ ]:
# Basic NumPy array operations
if has("likes", "impressions"):
    likes_arr = df["likes"].to_numpy(dtype=float)
    imp_arr = df["impressions"].to_numpy(dtype=float)

    print("Array shape       :", likes_arr.shape)
    print("Mean likes        :", np.nanmean(likes_arr).round(2))
    print("Median likes      :", np.nanmedian(likes_arr).round(2))
    print("Max / Min likes   :", np.nanmax(likes_arr), "/", np.nanmin(likes_arr))
    print("90th percentile   :", np.nanpercentile(likes_arr, 90).round(2))

    with np.errstate(divide="ignore", invalid="ignore"):
        like_ratio = likes_arr / imp_arr          # element-wise division
    print("Avg likes/impressions ratio:", np.nanmean(like_ratio[np.isfinite(like_ratio)]).round(4))

    # Min-max normalisation using vectorised NumPy
    norm_likes = (likes_arr - np.nanmin(likes_arr)) / (np.nanmax(likes_arr) - np.nanmin(likes_arr))
    print("Normalised likes range:", np.nanmin(norm_likes), "to", np.nanmax(norm_likes))

## Task 2 — Data Cleaning

### 2.1 Missing data

In [ ]:
# Detect missing values
print("Missing values per column (isnull):")
print(df.isnull().sum())
print("\nMissing % (isna):")
print((df.isna().mean() * 100).round(2))

In [ ]:
rows_before = len(df)

# a) Forward-fill / backward-fill for the date column (after sorting by date)
if "date" in df.columns:
    df = df.sort_values("date").reset_index(drop=True)
    df["date"] = df["date"].ffill().bfill()

# b) dropna(): rows with no engagement information at all are useless
eng_cols = [c for c in ["likes", "comments", "shares"] if c in df.columns]
if eng_cols:
    df = df.dropna(subset=eng_cols, how="all")

# c) fillna() with median for numeric columns
num_cols = df.select_dtypes(include="number").columns
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

# d) fillna() with mode for categorical columns (hashtags handled separately, ids skipped)
cat_cols = [c for c in df.select_dtypes(include=["object", "category"]).columns
            if c != "hashtags" and "id" not in c.split("_")]
for c in cat_cols:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].mode()[0])

if "hashtags" in df.columns:
    df["hashtags"] = df["hashtags"].fillna("")

print(f"Rows before: {rows_before} | after: {len(df)}")
print("Remaining missing values:", int(df.isnull().sum().sum()))

### 2.2 Duplicate handling

In [ ]:
print("Fully duplicated rows:", df.duplicated().sum())
id_col = next((c for c in df.columns if c in ("post_id", "id")), None)
if id_col:
    print(f"Duplicate {id_col}s:", df.duplicated(subset=[id_col]).sum())

before = len(df)
df = df.drop_duplicates()
if id_col:
    df = df.drop_duplicates(subset=[id_col], keep="first")
df = df.reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows. Shape now: {df.shape}")

### 2.3 Data formatting (types, categories, unrealistic values)

In [ ]:
# Standardise text categories (trim spaces + consistent case)
for c in ["post_type", "category", "device"]:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip().str.title()

if "country" in df.columns:
    df["country"] = (df["country"].astype(str).str.strip()
                     .apply(lambda s: s.upper() if len(s) <= 3 else s.title()))

# Gender labels -> Male / Female / Other
if "gender" in df.columns:
    gmap = {"m": "Male", "male": "Male", "man": "Male",
            "f": "Female", "female": "Female", "woman": "Female",
            "other": "Other", "non-binary": "Other", "nonbinary": "Other", "nb": "Other"}
    df["gender"] = df["gender"].astype(str).str.strip().str.lower().map(gmap).fillna("Other")
    print("Gender labels:", df["gender"].unique())

# Verified flag -> boolean + readable label
if "verified" in df.columns:
    df["verified"] = df["verified"].astype(str).str.strip().str.lower().isin(
        ["true", "yes", "y", "1", "verified"])
    df["verified_label"] = np.where(df["verified"], "Verified", "Not Verified")

In [ ]:
# Correct unrealistic values
metric_cols = [c for c in ["likes", "comments", "shares", "impressions", "followers", "watch_time"]
               if c in df.columns]

# 1) negative counts are impossible -> treat as missing and replace with the median
for c in metric_cols:
    bad = (df[c] < 0).sum()
    df.loc[df[c] < 0, c] = np.nan
    df[c] = df[c].fillna(df[c].median())
    print(f"{c}: fixed {bad} negative value(s)")

# 2) age outside a realistic range (13-90) -> median
if "age" in df.columns:
    bad_age = ((df["age"] < 13) | (df["age"] > 90)).sum()
    df.loc[(df["age"] < 13) | (df["age"] > 90), "age"] = np.nan
    df["age"] = df["age"].fillna(df["age"].median()).astype(int)
    print(f"age: fixed {bad_age} unrealistic value(s)")

# 3) likes can't exceed impressions
if has("likes", "impressions"):
    bad = (df["likes"] > df["impressions"]).sum()
    df["impressions"] = np.maximum(df["impressions"], df["likes"])
    print(f"likes > impressions: corrected {bad} row(s)")

# 4) cap extreme outliers in likes/comments/shares at the 99.5th percentile
for c in [c for c in ["likes", "comments", "shares"] if c in df.columns]:
    cap = df[c].quantile(0.995)
    n = (df[c] > cap).sum()
    df[c] = df[c].clip(upper=cap)
    print(f"{c}: capped {n} extreme value(s) at {cap:,.0f}")

### 2.4 Feature cleaning (hashtag count, sentiment labels)

In [ ]:
# Extract hashtag count
def count_tags(x):
    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return 0
    if "#" in s:
        return s.count("#")
    return len([t for t in re.split(r"[,\s;|]+", s) if t])

if "hashtags" in df.columns:
    df["hashtag_count"] = df["hashtags"].apply(count_tags)
elif "hashtag_count" not in df.columns:
    print("No hashtag information in the dataset")
print(df[["hashtag_count"]].describe().T if "hashtag_count" in df.columns else "")

# Clean sentiment labels -> Positive / Neutral / Negative
if "sentiment" in df.columns:
    smap = {"positive": "Positive", "pos": "Positive", "+": "Positive", "good": "Positive", "happy": "Positive",
            "negative": "Negative", "neg": "Negative", "-": "Negative", "bad": "Negative", "angry": "Negative",
            "neutral": "Neutral", "neu": "Neutral", "mixed": "Neutral", "none": "Neutral"}
    df["sentiment"] = (df["sentiment"].astype(str).str.strip().str.lower()
                       .map(smap).fillna("Neutral"))
    print(df["sentiment"].value_counts())

## Task 3 — Data Exploration using Pandas

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
display(df.head())
display(df.tail())

In [ ]:
df.info()
print()
print(df.dtypes)

In [ ]:
# Summary statistics
df.describe().T

In [ ]:
# Categorical distributions
cat_features = [c for c in ["post_type", "category", "country", "gender", "device", "sentiment", "verified_label"]
                if c in df.columns]
for c in cat_features:
    print(f"--- {c} | unique: {df[c].nunique()} ---")
    print(df[c].value_counts().head(10))
    print()

In [ ]:
# Correlation matrix of numeric fields
numeric_df = df.select_dtypes(include="number")
corr = numeric_df.corr()
corr.round(2)

In [ ]:
# groupby summaries
if has("post_type", "likes"):
    display(df.groupby("post_type")["likes"].agg(["count", "mean", "median"]).sort_values("mean", ascending=False))
if has("country", "impressions"):
    display(df.groupby("country")["impressions"].mean().sort_values(ascending=False).head(10).to_frame("avg_impressions"))
if has("category", "likes"):
    display(df.groupby("category")["likes"].mean().sort_values(ascending=False).to_frame("avg_likes"))

## Task 4 — Data Wrangling

In [ ]:
# New fields
# engagement_score: weighted interactions (a share is worth more than a like)
df["engagement_score"] = df["likes"] + 2 * df["comments"] + 3 * df["shares"]

# engagement_rate (%) if it is not already in the dataset
if "engagement_rate" not in df.columns:
    base = df["impressions"].replace(0, np.nan)
    df["engagement_rate"] = ((df["likes"] + df["comments"] + df["shares"]) / base * 100).fillna(0)

# log-transformed metrics (reduce skew)
for c in [c for c in ["likes", "comments", "shares", "impressions", "followers"] if c in df.columns]:
    df[f"log_{c}"] = np.log1p(df[c])

# time features from the date column
if "date" in df.columns:
    df["day_of_week"] = df["date"].dt.day_name()
    df["month"] = df["date"].dt.month_name()
    if "hour" not in df.columns and (df["date"].dt.hour != 0).any():
        df["hour"] = df["date"].dt.hour

df[["likes", "comments", "shares", "engagement_score", "engagement_rate", "hashtag_count"]
   if "hashtag_count" in df.columns else
   ["likes", "comments", "shares", "engagement_score", "engagement_rate"]].head()

In [ ]:
# merge: attach each category's average likes back onto every row
if has("category", "likes"):
    cat_avg = (df.groupby("category", as_index=False)["likes"].mean()
                 .rename(columns={"likes": "category_avg_likes"}))
    df = df.merge(cat_avg, on="category", how="left")
    df["above_category_avg"] = df["likes"] > df["category_avg_likes"]
    print("Posts above their category average:", df["above_category_avg"].mean().round(3) * 100, "%")

# concat: stack the top 5 and bottom 5 posts by engagement score
top_bottom = pd.concat([df.nlargest(5, "engagement_score").assign(group="Top 5"),
                        df.nsmallest(5, "engagement_score").assign(group="Bottom 5")])
top_bottom[["group", "likes", "comments", "shares", "engagement_score"]]

In [ ]:
# groupby summaries by post_type, country and sentiment
agg = {c: "mean" for c in ["likes", "comments", "shares", "impressions", "engagement_rate", "engagement_score"]
       if c in df.columns}
for key in ["post_type", "country", "sentiment"]:
    if key in df.columns:
        print(f"===== Average metrics by {key} =====")
        display(df.groupby(key).agg(agg).sort_values("engagement_rate", ascending=False).round(2))

## Task 5 — Statistical Analysis

In [ ]:
stat_cols = [c for c in ["likes", "comments", "shares", "watch_time", "engagement_rate", "followers"]
             if c in df.columns]

rows = {}
for c in stat_cols:
    s = df[c]
    rows[c] = {
        "mean": s.mean(), "median": s.median(), "mode": s.mode().iloc[0],
        "std": s.std(), "variance": s.var(),
        "min": s.min(), "p25": s.quantile(.25), "p50": s.quantile(.50), "p75": s.quantile(.75),
        "p90": s.quantile(.90), "p95": s.quantile(.95), "max": s.max(),
        "skewness": s.skew(), "kurtosis": s.kurt(),
    }
stats_table = pd.DataFrame(rows).T.round(2)
stats_table

**Reading the table:** a mean higher than the median and a positive skewness mean a few viral posts pull the average up (right-skewed). Kurtosis above 0 means heavy tails, i.e. more extreme values than a normal distribution.

## Task 6 — Data Visualization (Matplotlib, Seaborn, Plotly)

### 6.1 Matplotlib

In [ ]:
# 1. Scatter: likes vs impressions
if has("likes", "impressions"):
    plt.figure(figsize=(8, 5))
    plt.scatter(df["impressions"], df["likes"], alpha=0.4, s=12, color="steelblue")
    plt.title("Likes vs Impressions")
    plt.xlabel("Impressions"); plt.ylabel("Likes")
    plt.tight_layout(); plt.show()

In [ ]:
# 2. Line: daily engagement trend
if has("date", "engagement_score"):
    daily = df.groupby(df["date"].dt.date)["engagement_score"].sum()
    plt.figure(figsize=(12, 4))
    plt.plot(daily.index, daily.values, color="darkorange", linewidth=1)
    plt.title("Daily Engagement Trend (total engagement score)")
    plt.xlabel("Date"); plt.ylabel("Engagement score")
    plt.xticks(rotation=45)
    plt.tight_layout(); plt.show()

In [ ]:
# 3. Bar: posts by category
if has("category"):
    counts = df["category"].value_counts()
    plt.figure(figsize=(9, 5))
    plt.bar(counts.index, counts.values, color="seagreen")
    plt.title("Number of Posts by Category")
    plt.xlabel("Category"); plt.ylabel("Posts")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout(); plt.show()

In [ ]:
# 4. Pie: gender distribution
if has("gender"):
    g = df["gender"].value_counts()
    plt.figure(figsize=(6, 6))
    plt.pie(g.values, labels=g.index, autopct="%1.1f%%", startangle=90)
    plt.title("Gender Distribution")
    plt.tight_layout(); plt.show()

In [ ]:
# 5. Histogram: age
if has("age"):
    plt.figure(figsize=(8, 5))
    plt.hist(df["age"], bins=20, color="mediumpurple", edgecolor="black")
    plt.title("Age Distribution")
    plt.xlabel("Age"); plt.ylabel("Number of users")
    plt.tight_layout(); plt.show()

In [ ]:
# 6. Box: engagement rate
plt.figure(figsize=(6, 5))
plt.boxplot(df["engagement_rate"], vert=True, patch_artist=True,
            boxprops=dict(facecolor="lightcoral"))
plt.title("Engagement Rate (%)")
plt.ylabel("Engagement rate")
plt.tight_layout(); plt.show()

### 6.2 Seaborn

In [ ]:
# 7. Count plot: post type
if has("post_type"):
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df, x="post_type", order=df["post_type"].value_counts().index)
    plt.title("Posts by Post Type"); plt.xlabel("Post type"); plt.ylabel("Count")
    plt.xticks(rotation=30)
    plt.tight_layout(); plt.show()

In [ ]:
# 8. Bar plot: average likes by category
if has("category", "likes"):
    order = df.groupby("category")["likes"].mean().sort_values(ascending=False).index
    plt.figure(figsize=(9, 5))
    sns.barplot(data=df, x="category", y="likes", order=order, errorbar=None)
    plt.title("Average Likes by Category"); plt.xlabel("Category"); plt.ylabel("Average likes")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout(); plt.show()

In [ ]:
# 9. Violin: followers vs sentiment
if has("sentiment", "followers"):
    plt.figure(figsize=(8, 5))
    sns.violinplot(data=df, x="sentiment", y="followers", hue="sentiment", palette="Set2", legend=False)
    plt.title("Followers Distribution by Sentiment")
    plt.tight_layout(); plt.show()

In [ ]:
# 10. Pair plot: numeric features (random sample of 500 rows keeps it fast and readable)
pair_cols = [c for c in ["likes", "comments", "shares", "impressions", "engagement_rate", "followers"]
             if c in df.columns]
sample = df.sample(min(500, len(df)), random_state=42)
g = sns.pairplot(sample, vars=pair_cols, corner=True, plot_kws={"alpha": 0.5, "s": 15})
g.figure.suptitle("Pair Plot of Numeric Features", y=1.02)
plt.show()

In [ ]:
# 11. Heatmap: correlation matrix
corr_cols = [c for c in ["likes", "comments", "shares", "impressions", "watch_time", "engagement_rate",
                         "followers", "age", "hashtag_count", "engagement_score"] if c in df.columns]
plt.figure(figsize=(9, 7))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Matrix")
plt.tight_layout(); plt.show()

In [ ]:
# 12. Swarm plot: engagement vs device (sample of 400 points - swarm plots are slow on 5000)
if has("device", "engagement_rate"):
    sw = df.sample(min(400, len(df)), random_state=1)
    plt.figure(figsize=(9, 5))
    sns.swarmplot(data=sw, x="device", y="engagement_rate", hue="device", size=3, legend=False)
    plt.title("Engagement Rate by Device (sample of 400 posts)")
    plt.tight_layout(); plt.show()

### 6.3 Plotly (interactive)

In [ ]:
if PLOTLY_OK:
    # Interactive line chart: daily average engagement rate
    if has("date"):
        d = df.groupby(df["date"].dt.date)["engagement_rate"].mean().reset_index()
        fig = px.line(d, x="date", y="engagement_rate", title="Daily Average Engagement Rate")
        fig.show()

In [ ]:
if PLOTLY_OK and has("country"):
    # Interactive bar chart: average engagement rate by country
    c = df.groupby("country")["engagement_rate"].mean().sort_values(ascending=False).reset_index()
    fig = px.bar(c, x="country", y="engagement_rate", color="engagement_rate",
                 title="Average Engagement Rate by Country")
    fig.show()

In [ ]:
if PLOTLY_OK and has("followers", "likes"):
    # Interactive bubble chart: followers vs likes, bubble size = impressions
    s = df.sample(min(1000, len(df)), random_state=7)
    fig = px.scatter(s, x="followers", y="likes", size="impressions",
                     color="post_type" if "post_type" in s.columns else None,
                     hover_data=["engagement_rate"], opacity=0.6,
                     title="Followers vs Likes (bubble size = impressions)")
    fig.show()

## Final Insights

The cell below computes every finding directly from the cleaned data, so the numbers in the summary always match the analysis.

In [ ]:
E = "engagement_rate"
lines = []
add = lines.append

def mean_by(key, metric=E):
    return df.groupby(key)[metric].mean().sort_values(ascending=False)

add("## Content Performance")
if "post_type" in df:
    s = mean_by("post_type")
    add(f"- Post type with the highest average engagement rate: **{s.index[0]}** ({s.iloc[0]:.2f}%); lowest: {s.index[-1]} ({s.iloc[-1]:.2f}%).")
if "category" in df:
    s = mean_by("category")
    add(f"- Best-performing content category: **{s.index[0]}** ({s.iloc[0]:.2f}% avg engagement rate).")
if "country" in df:
    s = mean_by("country")
    add("- Top countries by average engagement rate: " + ", ".join(f"**{i}** ({v:.2f}%)" for i, v in s.head(3).items()) + ".")

add("\n## User Trends")
if "age" in df:
    bins = [12, 18, 25, 35, 45, 55, 100]
    labels = ["13-18", "19-25", "26-35", "36-45", "46-55", "56+"]
    ag = df.groupby(pd.cut(df["age"], bins=bins, labels=labels))[E].mean().dropna()
    r = df["age"].corr(df[E])
    add(f"- Age group with the highest engagement rate: **{ag.idxmax()}** ({ag.max():.2f}%); lowest: {ag.idxmin()} ({ag.min():.2f}%). Correlation between age and engagement rate: {r:.2f}.")
if "verified_label" in df:
    v = mean_by("verified_label")
    if len(v) == 2:
        diff = (v.get("Verified", np.nan) / v.get("Not Verified", np.nan) - 1) * 100
        add(f"- Verified accounts average {v.get('Verified', np.nan):.2f}% engagement vs {v.get('Not Verified', np.nan):.2f}% for non-verified ({diff:+.1f}% difference).")

add("\n## Behavioral Insights")
tkey = "hour" if "hour" in df else ("time_of_day" if "time_of_day" in df else None)
if tkey:
    t = df.groupby(tkey)["impressions"].mean().sort_values(ascending=False)
    add(f"- Best time for impressions ({tkey}): **{t.index[0]}** ({t.iloc[0]:,.0f} avg impressions).")
if "device" in df and "watch_time" in df:
    d = df.groupby("device")["watch_time"].mean().sort_values(ascending=False)
    add(f"- Device with the longest average watch time: **{d.index[0]}** ({d.iloc[0]:.1f}); shortest: {d.index[-1]} ({d.iloc[-1]:.1f}).")

add("\n## Sentiment Analysis")
if "sentiment" in df:
    s = mean_by("sentiment")
    add(f"- Sentiment with the highest average engagement rate: **{s.index[0]}** ({s.iloc[0]:.2f}%).")
    comp = df.groupby("sentiment")[[c for c in ["likes", "comments", "shares", E] if c in df]].mean().round(2)
    if "Positive" in comp.index:
        for lab in ["Negative", "Neutral"]:
            if lab in comp.index:
                pct = ((comp.loc[lab] / comp.loc["Positive"] - 1) * 100).round(1)
                add(f"- {lab} posts vs Positive posts: " + ", ".join(f"{k} {v:+.1f}%" for k, v in pct.items()) + ".")
    try:
        add("\n" + comp.to_markdown())
    except ImportError:
        add("\n" + comp.to_string())

summary = "\n".join(lines)
print(summary)
with open("final_insights.md", "w") as f:
    f.write("# Social Media Engagement Analytics - One-Page Summary\n\n" + summary)

## One-Page Summary (for submission)
Copy the findings printed above (also saved as `final_insights.md`) into your one-page summary and add a short **recommendation** for each section, e.g. which post type/category to prioritise, when to post, and how to handle negative-sentiment content.